# Practical: Google Colab, GitHub, and Landsat 8 Data Acquisition
## Visualising spectral bands and extracting a reflectance spectrum

**Suggested duration:** 2-3 hours  
**Platform:** Google Colab + GitHub + Google Earth Engine  
**Dataset:** USGS Landsat 8 Collection 2, Level 2, Tier 1 surface reflectance

### Learning outcomes
By the end of this practical, you should be able to:
1. Open a Jupyter notebook stored on GitHub in Google Colab.
2. Run, edit, and save a working copy of a Colab notebook.
3. Authenticate and initialise the Google Earth Engine Python API.
4. Search Landsat 8 scenes by area, date, and cloud cover.
5. Apply the Collection 2 surface-reflectance scale factor and cloud mask.
6. Display individual bands, true-colour and false-colour composites.
7. Extract surface reflectance at a selected location and plot a spectral signature.
8. Export a processed Landsat subset to Google Drive.

> **Important:** Run notebook cells in order. Replace `YOUR_PROJECT_ID` with an Earth Engine-enabled Google Cloud project that you are permitted to use.


## Part A - Working with a notebook from GitHub

### Option 1: Open the notebook directly in Colab
If the notebook is stored at:

`https://github.com/USERNAME/REPOSITORY/blob/main/Landsat_Colab_GitHub_Lab.ipynb`

replace the beginning with:

`https://colab.research.google.com/github/USERNAME/REPOSITORY/blob/main/Landsat_Colab_GitHub_Lab.ipynb`

You can also use **File > Open notebook > GitHub** in Colab.

### Option 2: Clone the entire GitHub repository into the Colab runtime
Use this when the notebook requires other files from the same repository.


In [ ]:
# OPTIONAL: clone a GitHub repository into the temporary Colab runtime
# Replace the URL with your course repository.
# !git clone https://github.com/USERNAME/REPOSITORY.git
# %cd REPOSITORY
# !ls


### Colab handling rules
- `Shift + Enter` runs the selected cell and moves to the next cell.
- A Colab runtime is temporary. Files created under `/content` can disappear when the runtime is reset.
- When you open a notebook from GitHub, save your own working copy before making major edits.
- Do not run code from an unknown repository unless you understand and trust what the notebook does.


## Part B - Install/import packages and initialise Earth Engine

In [ ]:
# geemap provides convenient interactive Earth Engine maps in notebooks.
%pip -q install -U geemap

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('Packages imported successfully.')


In [ ]:
# Authenticate and initialise Earth Engine.
# Replace this with the Google Cloud Project ID assigned for the practical.
PROJECT_ID = 'YOUR_PROJECT_ID'

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

print('Earth Engine initialised.')


## Part C - Define the study area

The example below uses a point near Thiruvananthapuram and creates a 20 km buffer. Change the longitude and latitude to study another location.

Earth Engine coordinates are entered as **[longitude, latitude]**.


In [ ]:
# Example centre point near Thiruvananthapuram, Kerala.
CENTER_LON = 76.9366
CENTER_LAT = 8.5241

centre = ee.Geometry.Point([CENTER_LON, CENTER_LAT])
aoi = centre.buffer(20_000).bounds()  # 20 km buffer, converted to a bounding rectangle
sample_point = centre

print('AOI created.')


## Part D - Search and acquire Landsat 8 data

We use the Earth Engine collection:

`LANDSAT/LC08/C02/T1_L2`

This is **Landsat 8 Collection 2, Level 2, Tier 1**. The optical bands are surface-reflectance products. We filter by location, date, and scene cloud cover, then select the least-cloudy scene from the search results.


In [ ]:
START_DATE = '2025-01-01'
END_DATE   = '2025-12-31'
MAX_CLOUD  = 60  # scene-level cloud cover percentage

raw_collection = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterBounds(aoi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt('CLOUD_COVER', MAX_CLOUD))
)

scene_count = raw_collection.size().getInfo()
print('Number of matching scenes:', scene_count)

if scene_count == 0:
    raise ValueError('No scenes found. Increase the date range or MAX_CLOUD.')


## Part E - Apply reflectance scaling and cloud masking

Landsat Collection 2 Level-2 optical bands are stored as scaled integers. For bands `SR_B1` to `SR_B7`, surface reflectance is calculated as:

\[
\rho = DN \times 0.0000275 - 0.2
\]

The `QA_PIXEL` bit field identifies dilated cloud, cirrus, cloud, cloud shadow, and snow. The function below masks those pixels and also removes radiometrically saturated pixels using `QA_RADSAT`.


In [ ]:
def preprocess_landsat8(image):
    # QA_PIXEL bits used here:
    # bit 1 = dilated cloud
    # bit 2 = cirrus
    # bit 3 = cloud
    # bit 4 = cloud shadow
    # bit 5 = snow
    qa = image.select('QA_PIXEL')

    clear_mask = (
        qa.bitwiseAnd(1 << 1).eq(0)
        .And(qa.bitwiseAnd(1 << 2).eq(0))
        .And(qa.bitwiseAnd(1 << 3).eq(0))
        .And(qa.bitwiseAnd(1 << 4).eq(0))
        .And(qa.bitwiseAnd(1 << 5).eq(0))
    )

    # Keep pixels that are not radiometrically saturated.
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply official Collection 2 scaling to optical surface-reflectance bands.
    optical = image.select('SR_B[1-7]').multiply(0.0000275).add(-0.2)

    return (
        image.addBands(optical, None, True)
        .updateMask(clear_mask)
        .updateMask(saturation_mask)
    )

# Choose the least-cloudy matching scene, then preprocess it.
raw_image = ee.Image(raw_collection.sort('CLOUD_COVER').first())
image = preprocess_landsat8(raw_image)

print('Product ID:', raw_image.get('LANDSAT_PRODUCT_ID').getInfo())
print('Acquisition date:', raw_image.date().format('YYYY-MM-dd').getInfo())
print('Scene cloud cover (%):', raw_image.get('CLOUD_COVER').getInfo())


## Part F - Landsat 8 optical bands used in this practical

| Band | Region | Approx. wavelength range (micrometres) | Common use |
|---|---|---:|---|
| SR_B1 | Coastal/Aerosol | 0.435-0.451 | Coastal/aerosol studies |
| SR_B2 | Blue | 0.452-0.512 | Water, haze, true colour |
| SR_B3 | Green | 0.533-0.590 | Vegetation/water contrast |
| SR_B4 | Red | 0.636-0.673 | Chlorophyll absorption, true colour |
| SR_B5 | NIR | 0.851-0.879 | Vegetation and land-water contrast |
| SR_B6 | SWIR1 | 1.566-1.651 | Moisture, soil and burn sensitivity |
| SR_B7 | SWIR2 | 2.107-2.294 | Moisture, geology and burn sensitivity |


## Part G - Visualise the Landsat scene

In [ ]:
Map = geemap.Map()
Map.centerObject(aoi, 10)

# True colour: Red, Green, Blue = B4, B3, B2
true_color = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],
    'min': 0.0,
    'max': 0.30,
}

# Colour infrared: NIR, Red, Green = B5, B4, B3
false_color = {
    'bands': ['SR_B5', 'SR_B4', 'SR_B3'],
    'min': 0.0,
    'max': 0.35,
}

Map.addLayer(image.clip(aoi), true_color, 'True colour (4-3-2)')
Map.addLayer(image.clip(aoi), false_color, 'False colour (5-4-3)', False)
Map.addLayer(aoi, {'color': 'yellow'}, 'AOI', False)
Map.addLayer(sample_point, {'color': 'red'}, 'Sample point')
Map


### Display individual bands
Turn layers on and off in the map layer control and compare the appearance of the same surface at different wavelengths.


In [ ]:
band_names = [
    ('SR_B1', 'Coastal aerosol'),
    ('SR_B2', 'Blue'),
    ('SR_B3', 'Green'),
    ('SR_B4', 'Red'),
    ('SR_B5', 'NIR'),
    ('SR_B6', 'SWIR1'),
    ('SR_B7', 'SWIR2'),
]

BandMap = geemap.Map()
BandMap.centerObject(aoi, 10)

for band, label in band_names:
    BandMap.addLayer(
        image.select(band).clip(aoi),
        {'min': 0.0, 'max': 0.35},
        label,
        False,
    )

# Show the red band by default.
BandMap.addLayer(image.select('SR_B4').clip(aoi), {'min': 0.0, 'max': 0.35}, 'Red - active')
BandMap.addLayer(sample_point, {'color': 'red'}, 'Sample point')
BandMap


## Part H - Extract and plot a spectral signature

A spectral signature is a plot of reflectance as a function of wavelength. Here we average the clear Landsat pixels within a 45 m radius of the sample point to reduce sensitivity to a single pixel.


In [ ]:
bands = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']

# Approximate band-centre wavelengths in micrometres.
wavelength_um = [0.443, 0.482, 0.562, 0.655, 0.865, 1.609, 2.201]

sample_area = sample_point.buffer(45)

spectrum = (
    image.select(bands)
    .reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=sample_area,
        scale=30,
        maxPixels=1_000_000,
    )
    .getInfo()
)

reflectance = [spectrum.get(b, np.nan) for b in bands]

spectral_table = pd.DataFrame({
    'Band': bands,
    'Wavelength_um': wavelength_um,
    'Surface_reflectance': reflectance,
})

spectral_table


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(wavelength_um, reflectance, marker='o')
plt.xlabel('Wavelength (micrometres)')
plt.ylabel('Surface reflectance')
plt.title('Landsat 8 spectral signature at selected location')
plt.grid(True, alpha=0.3)
plt.show()


### Interpret your spectrum
Ask:
1. Which band has the lowest reflectance?
2. Which band has the highest reflectance?
3. Is there a strong increase from red to NIR?
4. Does the spectrum look more like vegetation, water, bare soil, or built-up land?
5. How might cloud, cloud shadow, mixed pixels, or atmospheric-correction errors alter the spectrum?


## Part I - Optional: calculate NDVI

\[
NDVI = \frac{NIR - Red}{NIR + Red}
\]

For Landsat 8, NIR is `SR_B5` and red is `SR_B4`.


In [ ]:
ndvi = image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')

NDVIMap = geemap.Map()
NDVIMap.centerObject(aoi, 10)
NDVIMap.addLayer(ndvi.clip(aoi), {'min': -1, 'max': 1}, 'NDVI')
NDVIMap


## Part J - Export the processed Landsat bands to Google Drive

This starts an Earth Engine export task. The result will be a GeoTIFF in the selected Google Drive folder after the server-side task finishes.


In [ ]:
export_image = image.select(bands).clip(aoi)

task = ee.batch.Export.image.toDrive(
    image=export_image,
    description='Landsat8_SR_Thiruvananthapuram',
    folder='EarthEngine',
    fileNamePrefix='Landsat8_SR_example',
    region=aoi,
    scale=30,
    maxPixels=1e9,
    fileFormat='GeoTIFF',
)

# Uncomment the next line when you are ready to export.
# task.start()

print('Export prepared. Uncomment task.start() to submit it.')


## Part K - Exercises for submission

1. Change the study area to a location assigned by the instructor.
2. Change the date range and record the number of Landsat scenes found.
3. Record the product ID, acquisition date, and scene cloud-cover percentage of the selected scene.
4. Submit screenshots of:
   - true-colour composite (4-3-2),
   - false-colour composite (5-4-3),
   - one visible band,
   - NIR band,
   - one SWIR band.
5. Select at least **three surface types** (for example vegetation, water, and built-up/bare soil), extract a spectral signature for each, and compare them on one graph.
6. Explain why Collection 2 scale factors must be applied before interpreting reflectance values.
7. Explain why cloud and cloud-shadow masking matters for spectral analysis.
8. Optional: calculate and interpret NDVI for the study area.

### Suggested submission
- Completed `.ipynb` notebook.
- One-page result sheet containing the requested figures and a short interpretation.


## Troubleshooting

**`EEException: Project ... not registered`**  
Use a Google Cloud project that is registered/enabled for Earth Engine and for which your account has permission.

**Authentication repeatedly fails**  
Check that Colab is signed into the intended Google account and rerun `ee.Authenticate()`.

**No Landsat scenes found**  
Increase the date range or `MAX_CLOUD`, or check the AOI coordinates.

**Spectrum contains `nan` values**  
The point may be cloudy, shadowed, outside valid data, or masked. Move the sample point or choose another scene/date.

**The map appears very dark/bright**  
Adjust the visualization `min` and `max`. These values affect display only; they do not change the data.

**A cloned repository disappears**  
The `/content` runtime is temporary. Save persistent work to Drive or GitHub.


## Reference sources
- Google Colab FAQ: https://research.google.com/colaboratory/faq.html
- Google Earth Engine authentication: https://developers.google.com/earth-engine/guides/auth
- Earth Engine Landsat 8 Collection 2 Level 2 catalog: https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2
- Earth Engine Cloud project transition: https://developers.google.com/earth-engine/guides/transition_to_cloud_projects
